# Construcción del dataset raw para detección de similitud de código

Este notebook prepara la primera versión del dataset que se usará para entrenar y validar un modelo de comparación entre programas. En esta etapa se conserva el código fuente en formato crudo, es decir, sin limpiar, normalizar, tokenizar ni extraer características.

El objetivo principal es transformar una colección de archivos de código y archivos de referencia qrel en tablas con pares de programas. Cada par queda etiquetado con label igual a 1 cuando representa una relación positiva, o label igual a 0 cuando se genera como ejemplo negativo.

Trabajar primero con una versión raw ayuda a separar responsabilidades: este notebook organiza los datos y deja el preprocesamiento lingüístico para etapas posteriores.


## 1. Importación de librerías

En esta celda se cargan herramientas para manejar rutas, expresiones regulares, aleatoriedad, tablas y particiones de datos.

Conceptos importantes:

- Path permite construir rutas de forma segura y legible.
- pandas representa los archivos y pares como DataFrames, estructuras tabulares útiles para filtrar, unir y exportar datos.
- train_test_split divide el dataset en subconjuntos de entrenamiento y validación.


In [1]:
import os
import re
import random
import pandas as pd
from pathlib import Path
from itertools import combinations
from sklearn.model_selection import train_test_split

## 2. Definición de rutas del proyecto

Aquí se declaran las carpetas donde están los datos originales y donde se guardarán los archivos procesados. Centralizar estas rutas evita repetir cadenas en todo el notebook.

La carpeta data/processed se crea automáticamente para que las exportaciones posteriores no fallen si todavía no existe.


In [2]:
DATASET_PATH = Path("data")

TRAIN_PATH = DATASET_PATH / "train"
TEST_PATH = DATASET_PATH / "test"

PROCESSED_PATH = DATASET_PATH / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

## 3. Exploración inicial del conjunto de entrenamiento

Antes de construir el dataset conviene inspeccionar qué carpetas y archivos existen. Esta revisión confirma que el notebook apunta al lugar correcto y que el conjunto de entrenamiento contiene los lenguajes esperados.


In [21]:
print("Contenido de training:")
for item in TRAIN_PATH.iterdir():
    print("-", item.name)

Contenido de training:
- c
- java
- SOCO14-c.qrel
- SOCO14-java.qrel


### Conteo de archivos C y Java

Se listan los archivos fuente del entrenamiento por lenguaje. El conteo sirve como comprobación básica de disponibilidad y balance: si una carpeta está vacía o tiene muchos menos archivos de lo esperado, el dataset generado podría quedar sesgado o incompleto.


In [22]:
c_files = list((TRAIN_PATH / "c").glob("*.c"))
java_files = list((TRAIN_PATH / "java").glob("*.java"))

print("Archivos C:", len(c_files))
print("Archivos Java:", len(java_files))

Archivos C: 79
Archivos Java: 259


## 4. Exploración del conjunto de prueba

El conjunto de prueba puede tener una estructura distinta al entrenamiento, por ejemplo organizado por lenguaje y escenario. Por eso se revisa su contenido por separado antes de cargarlo de forma automática.


In [23]:
print("Contenido de test:")
for item in TEST_PATH.iterdir():
    print("-", item.name)

Contenido de test:
- c
- java
- README
- soco14-test-c-update.qrel
- soco14-test-java-update.qrel


### Conteo por lenguaje y escenario en test

Esta celda recorre las subcarpetas del conjunto de prueba y cuenta cuántos archivos hay en cada escenario. Un escenario representa una agrupación del problema, normalmente asociada a una familia de ejercicios, casos o condiciones de evaluación.


In [24]:
for language_folder in TEST_PATH.iterdir():
    if language_folder.is_dir():
        print("\nLenguaje:", language_folder.name)
        for scenario in language_folder.iterdir():
            if scenario.is_dir():
                n_files = len([f for f in scenario.iterdir() if f.is_file()])
                print("  ", scenario.name, "->", n_files, "archivos")


Lenguaje: c
   A1 -> 4683 archivos
   A2 -> 5195 archivos
   B1 -> 4303 archivos
   B2 -> 3873 archivos
   C1 -> 300 archivos
   C2 -> 145 archivos

Lenguaje: java
   A1 -> 3238 archivos
   A2 -> 3093 archivos
   B1 -> 3268 archivos
   B2 -> 2166 archivos
   C1 -> 124 archivos
   C2 -> 88 archivos


## 5. Lectura de código fuente en formato raw

La función read_raw_code lee cada archivo como texto completo. Se usa codificación UTF-8 con tolerancia a errores para reducir problemas por caracteres especiales o codificaciones mixtas.

En esta etapa no se aplica limpieza porque interesa conservar la forma original del código: comentarios, espacios, nombres de variables y formato. Esa decisión permite comparar después diferentes estrategias de preprocesamiento sin volver a cargar los archivos originales.


In [25]:
def read_raw_code(file_path):
    """
    Lee un archivo de código fuente como texto crudo.
    No aplica limpieza, tokenización ni normalización.
    """
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        print(f"Error leyendo {file_path}: {e}")
        return None

## 6. Carga del conjunto de entrenamiento raw

Esta función recorre las carpetas de entrenamiento para C y Java, lee cada archivo y crea un registro tabular por programa.

Cada fila guarda metadatos como split, language, file_name, file_id, file_path y code_raw. Esta tabla es la base para construir pares de programas más adelante.


In [26]:
def load_train_codes_raw(train_path):
    records = []

    language_configs = {
        "c": {
            "folder": "c",
            "extension": ".c"
        },
        "java": {
            "folder": "java",
            "extension": ".java"
        }
    }

    for language, config in language_configs.items():
        folder_path = train_path / config["folder"]

        if not folder_path.exists():
            print(f"No existe la carpeta: {folder_path}")
            continue

        for file_path in sorted(folder_path.glob(f"*{config['extension']}")):
            code = read_raw_code(file_path)

            if code is None:
                continue

            records.append({
                "split": "train",
                "language": language,
                "scenario": "train",
                "file_name": file_path.name,
                "file_id": file_path.stem,
                "file_path": str(file_path),
                "code_raw": code
            })

    return pd.DataFrame(records)

### Ejecución y vista previa del entrenamiento

Se carga el DataFrame de entrenamiento y se muestra su tamaño. La vista previa ayuda a verificar que las columnas esperadas existen y que code_raw contiene texto de código.


In [28]:
train_codes_raw = load_train_codes_raw(TRAIN_PATH)

print(train_codes_raw.shape)
train_codes_raw.head()

(338, 7)


,split,language,scenario,file_name,file_id,file_path,code_raw
0,train,c,train,000.c,000,data\train\c\000.c,#include<stdio.h>\n#include<string.h>\n#includ...
1,train,c,train,001.c,001,data\train\c\001.c,#include<stdio.h>\n#include<string.h>\n#includ...
2,train,c,train,002.c,002,data\train\c\002.c,#include<stdio.h>\n#include<string.h>\n#includ...
3,train,c,train,003.c,003,data\train\c\003.c,"\n\n#include ""stdio.h""\n#include ""stdlib.h""\n\..."
4,train,c,train,004.c,004,data\train\c\004.c,\n\n\n\n\n\n\n\n\n#include<stdio.h>\n#include<...


### Distribución por lenguaje

El conteo por language permite verificar cuántos archivos hay por lenguaje. Esta información es importante porque la generación de pares negativos se hará dentro del mismo lenguaje, evitando comparar artificialmente programas de lenguajes distintos.


In [29]:
train_codes_raw["language"].value_counts()

language
java    259
c        79
Name: count, dtype: int64

## 7. Detección de lenguaje y escenario en test

El conjunto de prueba puede contener carpetas o extensiones variadas. Por eso se definen funciones auxiliares para inferir el lenguaje del archivo usando la ruta y la extensión, y el escenario usando patrones como A1 o B2.

La detección por reglas es suficiente aquí porque la estructura del dataset ya contiene señales claras en nombres de carpetas y archivos.


In [30]:
VALID_CODE_EXTENSIONS = [".c", ".cpp", ".cc", ".cxx", ".java"]

def detect_test_language(file_path):
    parts = [p.lower() for p in file_path.parts]

    if "java" in parts:
        return "java"

    if "c_cpp" in parts or "c" in parts or "cpp" in parts:
        return "c_cpp"

    if file_path.suffix.lower() == ".java":
        return "java"

    if file_path.suffix.lower() in [".c", ".cpp", ".cc", ".cxx"]:
        return "c_cpp"

    return "unknown"


def detect_test_scenario(file_path):
    for part in file_path.parts:
        if re.fullmatch(r"[A-Z][0-9]", part):
            return part

    match = re.match(r"([A-Z][0-9])", file_path.stem)
    if match:
        return match.group(1)

    return "unknown"

## 8. Carga del conjunto de prueba raw

Esta función recorre recursivamente data/test, descarta archivos que no son código fuente útil para este paso, como qrel o readme, y guarda cada programa en un DataFrame con la misma idea general usada para entrenamiento.

Aunque el modelo se entrenará con pares del conjunto de entrenamiento, preparar también los archivos de prueba en formato raw deja listo el material para evaluaciones posteriores.


In [37]:
def load_test_codes_raw(test_path):
    records = []

    for file_path in test_path.rglob("*"):
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() == ".qrel" or file_path.name.lower() == "readme":
            continue

        code = read_raw_code(file_path)

        if code is None:
            continue

        records.append({
            "split": "test",
            "language": detect_test_language(file_path),
            "scenario": detect_test_scenario(file_path),
            "file_name": file_path.name,
            "file_id": file_path.stem,
            "file_path": str(file_path),
            "code_raw": code
        })

    return pd.DataFrame(records)

### Ejecución y vista previa del test

Se carga la tabla de archivos de prueba y se muestra una muestra inicial para comprobar que la detección de lenguaje, escenario y ruta funciona correctamente.


In [38]:
test_codes_raw = load_test_codes_raw(TEST_PATH)

print(test_codes_raw.shape)
test_codes_raw.head()

(30476, 7)


,split,language,scenario,file_name,file_id,file_path,code_raw
0,test,java,A1,A10000,A10000,data\test\java\A1\A10000,import java.io.*;\n\nclass goodance\n{\n\nstat...
1,test,java,A1,A10001,A10001,data\test\java\A1\A10001,import java.io.*;\nimport java.util.*;\n\npubl...
2,test,java,A1,A10002,A10002,data\test\java\A1\A10002,package qualification;\n\nimport java.util.Arr...
3,test,java,A1,A10003,A10003,data\test\java\A1\A10003,package qualification.b;\n\nimport java.util.S...
4,test,java,A1,A10004,A10004,data\test\java\A1\A10004,package qualificationRound;\n\nimport java.io....


### Distribución de test por lenguaje y escenario

Agrupar por language y scenario ayuda a revisar la composición del conjunto de prueba. Esta comprobación permite detectar escenarios vacíos, mal clasificados o con nombres inesperados.


In [39]:
test_codes_raw.groupby(["language", "scenario"]).size()

language  scenario
c_cpp     A1          4683
          A2          5195
          B1          4303
          B2          3873
          C1           300
          C2           145
java      A1          3238
          A2          3093
          B1          3268
          B2          2166
          C1           124
          C2            88
dtype: int64

## 9. Exportación de archivos raw individuales

Se guardan las tablas de programas individuales en data/processed. Estos archivos son salidas intermedias: no contienen pares todavía, pero permiten reutilizar la carga de código sin repetir el recorrido de carpetas.


In [41]:
train_codes_raw.to_csv(PROCESSED_PATH / "00_train_codes_raw.csv", index=False)
test_codes_raw.to_csv(PROCESSED_PATH / "00_test_codes_raw.csv", index=False)

## 10. Inspección de archivos qrel

Los archivos qrel contienen relaciones de referencia entre programas. En este contexto se usan para identificar pares positivos, es decir, pares de archivos que deben considerarse similares o relacionados.

Se define una función de vista previa para imprimir algunas líneas y entender el formato real antes de parsearlo automáticamente.


In [42]:
def preview_file(path, n=10):
    print(f"Archivo: {path}")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(repr(line.strip()))

### Vista previa de los qrel de entrenamiento

Se imprimen las primeras líneas de los archivos qrel de C y Java. Esta comprobación permite confirmar si las relaciones vienen como nombres de archivo completos o como identificadores numéricos.


In [43]:
preview_file(TRAIN_PATH / "SOCO14-c.qrel", n=10)
preview_file(TRAIN_PATH / "SOCO14-java.qrel", n=10)

Archivo: data\train\SOCO14-c.qrel
'005.c 006.c'
'007.c 032.c'
'007.c 043.c'
'007.c 050.c'
'008.c 041.c'
'009.c 045.c'
'013.c 046.c'
'017.c 018.c'
'026.c 064.c'
'026.c 050.c'
Archivo: data\train\SOCO14-java.qrel
'003.java 004.java'
'005.java 006.java'
'008.java 010.java'
'014.java 021.java'
'015.java 023.java'
'016.java 024.java'
'017.java 022.java'
'030.java 032.java'
'033.java 034.java'
'042.java 044.java'


## 11. Carga flexible de pares positivos

La función load_qrel_flexible transforma el contenido de un archivo qrel en una tabla de pares positivos.

Se implementa de forma flexible porque los qrel pueden representar archivos con nombres explícitos, como 000.c o 025.java, o con identificadores numéricos, que luego se convierten a nombres con ceros a la izquierda.

Cada relación válida se guarda con label igual a 1 porque representa una coincidencia positiva definida por el dataset.


In [44]:
def load_qrel_flexible(qrel_path, language):
    """
    Carga pares positivos desde archivos .qrel.
    Intenta detectar nombres como 000.c, 025.java
    o IDs numéricos como 000, 025.
    """
    rows = []

    extension = ".java" if language == "java" else ".c"

    with open(qrel_path, "r", encoding="utf-8", errors="ignore") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            parts = line.split()

            file_candidates = []

            for p in parts:
                p_clean = p.strip()

                if p_clean.endswith(".java") or p_clean.endswith(".c"):
                    file_candidates.append(p_clean)

            if len(file_candidates) < 2:
                numeric_candidates = [
                    p for p in parts
                    if re.fullmatch(r"\d+", p)
                ]

                if len(numeric_candidates) >= 2:
                    file_candidates = [
                        numeric_candidates[0].zfill(3) + extension,
                        numeric_candidates[1].zfill(3) + extension
                    ]

            if len(file_candidates) >= 2:
                rows.append({
                    "language": language,
                    "file_1": file_candidates[0],
                    "file_2": file_candidates[1],
                    "label": 1,
                    "source_qrel": str(qrel_path),
                    "line_number": line_number
                })

    return pd.DataFrame(rows)

### Unión de pares positivos de C y Java

Se leen por separado las relaciones positivas de C y Java y luego se unen en una sola tabla. Mantener el campo language permite analizar y filtrar los pares por lenguaje más adelante.


In [45]:
c_qrel = TRAIN_PATH / "SOCO14-c.qrel"
java_qrel = TRAIN_PATH / "SOCO14-java.qrel"

c_positive_raw = load_qrel_flexible(c_qrel, "c")
java_positive_raw = load_qrel_flexible(java_qrel, "java")

positive_pairs_raw = pd.concat(
    [c_positive_raw, java_positive_raw],
    ignore_index=True
)

print(positive_pairs_raw.shape)
positive_pairs_raw.head()

(110, 6)


,language,file_1,file_2,label,source_qrel,line_number
0,c,005.c,006.c,1,data\train\SOCO14-c.qrel,1
1,c,007.c,032.c,1,data\train\SOCO14-c.qrel,2
2,c,007.c,043.c,1,data\train\SOCO14-c.qrel,3
3,c,007.c,050.c,1,data\train\SOCO14-c.qrel,4
4,c,008.c,041.c,1,data\train\SOCO14-c.qrel,5


### Conteo de pares positivos por lenguaje

Este conteo revisa si las relaciones positivas están distribuidas entre los lenguajes esperados y ayuda a detectar problemas de lectura en algún archivo qrel.


In [46]:
positive_pairs_raw["language"].value_counts()

language
java    84
c       26
Name: count, dtype: int64

## 12. Validación de referencias contra archivos disponibles

Antes de unir código con pares, se comprueba que los nombres de archivo mencionados en los qrel existan realmente en los archivos cargados. Esta validación evita crear ejemplos con código faltante.


In [47]:
available_train_files = set(train_codes_raw["file_name"])

missing_file_1 = positive_pairs_raw[
    ~positive_pairs_raw["file_1"].isin(available_train_files)
]

missing_file_2 = positive_pairs_raw[
    ~positive_pairs_raw["file_2"].isin(available_train_files)
]

print("Faltantes en file_1:", len(missing_file_1))
print("Faltantes en file_2:", len(missing_file_2))

Faltantes en file_1: 0
Faltantes en file_2: 0


## 13. Asociación de pares positivos con su código fuente

Se crean mapas por nombre de archivo para recuperar rápidamente el código, la ruta y el lenguaje de cada programa. Estos diccionarios hacen más eficiente la unión porque evitan buscar manualmente cada archivo dentro del DataFrame.


In [48]:
code_map = train_codes_raw.set_index("file_name")["code_raw"].to_dict()
path_map = train_codes_raw.set_index("file_name")["file_path"].to_dict()
language_map = train_codes_raw.set_index("file_name")["language"].to_dict()

### Enriquecimiento de pares positivos

Los mapas se aplican a los pares positivos para añadir el código y las rutas de ambos archivos. Este paso convierte una relación entre nombres de archivo en un ejemplo completo que puede alimentar un modelo.


In [49]:
positive_pairs_with_code = positive_pairs_raw.copy()

positive_pairs_with_code["code_1_raw"] = positive_pairs_with_code["file_1"].map(code_map)
positive_pairs_with_code["code_2_raw"] = positive_pairs_with_code["file_2"].map(code_map)

positive_pairs_with_code["path_1"] = positive_pairs_with_code["file_1"].map(path_map)
positive_pairs_with_code["path_2"] = positive_pairs_with_code["file_2"].map(path_map)

positive_pairs_with_code["language_1"] = positive_pairs_with_code["file_1"].map(language_map)
positive_pairs_with_code["language_2"] = positive_pairs_with_code["file_2"].map(language_map)

positive_pairs_with_code.head()

,language,file_1,file_2,label,source_qrel,line_number,code_1_raw,code_2_raw,path_1,path_2,language_1,language_2
0,c,005.c,006.c,1,data\train\SOCO14-c.qrel,1,\n\n\n\n\n\n\n\n\n\n\n\n#include<stdio.h>\n#in...,\n\n\n\n\n\n\n\n\n\n#include<stdio.h>\n#includ...,data\train\c\005.c,data\train\c\006.c,c,c
1,c,007.c,032.c,1,data\train\SOCO14-c.qrel,2,#include <sys/time.h>\n#include <strings.h>\n#...,\n\n#include <stdio.h>\n#include <stdlib.h>\n#...,data\train\c\007.c,data\train\c\032.c,c,c
2,c,007.c,043.c,1,data\train\SOCO14-c.qrel,3,#include <sys/time.h>\n#include <strings.h>\n#...,\n\n\n#include <stdio.h>\n#include <stdlib.h>\...,data\train\c\007.c,data\train\c\043.c,c,c
3,c,007.c,050.c,1,data\train\SOCO14-c.qrel,4,#include <sys/time.h>\n#include <strings.h>\n#...,#include <stdio.h>\n#include <stdlib.h>\n#incl...,data\train\c\007.c,data\train\c\050.c,c,c
4,c,008.c,041.c,1,data\train\SOCO14-c.qrel,5,#include <stdio.h>\n#include <stdlib.h>\n#incl...,#include <strings.h>\n#include <string.h>\n#in...,data\train\c\008.c,data\train\c\041.c,c,c


### Limpieza y normalización de pares positivos

Se eliminan pares cuyo código no pudo encontrarse y se conserva únicamente la comparación dentro del mismo lenguaje. Esto es importante porque el objetivo es aprender similitud o plagio entre soluciones comparables, no diferencias entre lenguajes.

También se reorganizan las columnas para dejar una estructura clara y consistente.


In [50]:
positive_pairs_with_code = positive_pairs_with_code.dropna(
    subset=["code_1_raw", "code_2_raw"]
).reset_index(drop=True)

positive_pairs_with_code = positive_pairs_with_code[
    positive_pairs_with_code["language_1"] == positive_pairs_with_code["language_2"]
].reset_index(drop=True)

positive_pairs_with_code["scenario"] = "train"

positive_pairs_with_code = positive_pairs_with_code[
    [
        "language",
        "scenario",
        "file_1",
        "file_2",
        "path_1",
        "path_2",
        "code_1_raw",
        "code_2_raw",
        "label"
    ]
]

print(positive_pairs_with_code.shape)
positive_pairs_with_code.head()

(110, 9)


,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label
0,c,train,005.c,006.c,data\train\c\005.c,data\train\c\006.c,\n\n\n\n\n\n\n\n\n\n\n\n#include<stdio.h>\n#in...,\n\n\n\n\n\n\n\n\n\n#include<stdio.h>\n#includ...,1
1,c,train,007.c,032.c,data\train\c\007.c,data\train\c\032.c,#include <sys/time.h>\n#include <strings.h>\n#...,\n\n#include <stdio.h>\n#include <stdlib.h>\n#...,1
2,c,train,007.c,043.c,data\train\c\007.c,data\train\c\043.c,#include <sys/time.h>\n#include <strings.h>\n#...,\n\n\n#include <stdio.h>\n#include <stdlib.h>\...,1
3,c,train,007.c,050.c,data\train\c\007.c,data\train\c\050.c,#include <sys/time.h>\n#include <strings.h>\n#...,#include <stdio.h>\n#include <stdlib.h>\n#incl...,1
4,c,train,008.c,041.c,data\train\c\008.c,data\train\c\041.c,#include <stdio.h>\n#include <stdlib.h>\n#incl...,#include <strings.h>\n#include <string.h>\n#in...,1


### Exportación de pares positivos

Los pares positivos se guardan como una salida intermedia. Esto permite revisar, reutilizar o auditar las relaciones positivas sin regenerarlas desde los qrel.


In [51]:
positive_pairs_with_code.to_csv(
    PROCESSED_PATH / "01_train_positive_pairs_raw.csv",
    index=False
)

## 14. Preparación para generar pares negativos

Se construye positive_set con los pares positivos ya conocidos. Cada par se ordena internamente para que (A, B) y (B, A) se consideren el mismo par.

Este conjunto se usará para evitar que un par positivo sea seleccionado por error como negativo.


In [53]:
positive_set = set()

for _, row in positive_pairs_with_code.iterrows():
    pair_key = tuple(sorted([row["file_1"], row["file_2"]]))
    positive_set.add(pair_key)

len(positive_set)

110

## 15. Generación de pares negativos

Los pares negativos son ejemplos con label igual a 0. Se generan tomando dos archivos aleatorios del mismo lenguaje que no aparezcan en positive_set.

Conceptualmente, este paso crea una clase de contraste para el modelo: además de ver pares similares, el modelo necesita ver pares que no están marcados como similares. Se genera la misma cantidad de negativos que de positivos para producir un dataset balanceado por etiqueta.

La semilla seed=42 ayuda a que el muestreo sea reproducible.


In [54]:
def generate_negative_pairs_raw(train_codes_raw, positive_set, n_negatives, seed=42):
    random.seed(seed)

    negatives = []
    used_pairs = set()

    grouped = list(train_codes_raw.groupby("language"))

    attempts = 0
    max_attempts = n_negatives * 100

    while len(negatives) < n_negatives and attempts < max_attempts:
        attempts += 1

        language, group = random.choice(grouped)

        if len(group) < 2:
            continue

        sample = group.sample(2, random_state=None)

        row_1 = sample.iloc[0]
        row_2 = sample.iloc[1]

        file_1 = row_1["file_name"]
        file_2 = row_2["file_name"]

        pair_key = tuple(sorted([file_1, file_2]))

        if pair_key in positive_set:
            continue

        if pair_key in used_pairs:
            continue

        used_pairs.add(pair_key)

        negatives.append({
            "language": language,
            "scenario": "train",
            "file_1": file_1,
            "file_2": file_2,
            "path_1": row_1["file_path"],
            "path_2": row_2["file_path"],
            "code_1_raw": row_1["code_raw"],
            "code_2_raw": row_2["code_raw"],
            "label": 0
        })

    return pd.DataFrame(negatives)

### Ejecución de la generación negativa

Se calcula cuántos pares positivos existen y se solicita la misma cantidad de pares negativos. Luego se muestran tamaños y una vista previa para confirmar que ambos grupos están equilibrados.


In [55]:
n_positives = len(positive_pairs_with_code)

negative_pairs_raw = generate_negative_pairs_raw(
    train_codes_raw=train_codes_raw,
    positive_set=positive_set,
    n_negatives=n_positives,
    seed=42
)

print("Positivos:", positive_pairs_with_code.shape)
print("Negativos:", negative_pairs_raw.shape)
negative_pairs_raw.head()

Positivos: (110, 9)
Negativos: (110, 9)


,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label
0,c,train,071.c,013.c,data\train\c\071.c,data\train\c\013.c,\n\n#include<stdio.h>\n#include<strings.h>\n#i...,#include<stdio.h>\n#include<stdlib.h>\n#includ...,0
1,c,train,000.c,018.c,data\train\c\000.c,data\train\c\018.c,#include<stdio.h>\n#include<string.h>\n#includ...,#include <string.h>\n#include <stdlib.h>\n#inc...,0
2,java,train,124.java,019.java,data\train\java\124.java,data\train\java\019.java,\n\nimport java.text.*; \nimport java.util.*;...,\n\n\n\nimport java.util.*;\nimport java.util....,0
3,c,train,031.c,032.c,data\train\c\031.c,data\train\c\032.c,#include<stdio.h>\n#include<stdlib.h>\n#includ...,\n\n#include <stdio.h>\n#include <stdlib.h>\n#...,0
4,c,train,009.c,019.c,data\train\c\009.c,data\train\c\019.c,#include <stdio.h>\n#include <stdlib.h>\n#inc...,\n\n \n\t \n\n\n#include<stdio...,0


## 16. Construcción del dataset de pares de entrenamiento

Se unen pares positivos y negativos en una sola tabla, se mezclan aleatoriamente y se asigna un identificador único pair_id a cada ejemplo.

Mezclar los pares evita que todos los positivos o negativos queden agrupados por orden de creación. El identificador facilita rastrear ejemplos concretos durante depuración, entrenamiento o análisis de errores.


In [56]:
train_pairs_raw = pd.concat(
    [positive_pairs_with_code, negative_pairs_raw],
    ignore_index=True
)

train_pairs_raw = train_pairs_raw.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

train_pairs_raw["pair_id"] = [
    f"train_pair_{i:06d}" for i in range(len(train_pairs_raw))
]

train_pairs_raw = train_pairs_raw[
    [
        "pair_id",
        "language",
        "scenario",
        "file_1",
        "file_2",
        "path_1",
        "path_2",
        "code_1_raw",
        "code_2_raw",
        "label"
    ]
]

train_pairs_raw.head()

,pair_id,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label
0,train_pair_000000,java,train,180.java,001.java,data\train\java\180.java,data\train\java\001.java,import java.io.*;\nimport java.net.*;\nimport ...,\n\n\nimport java.io.*;\n\nclass WatchDog {\n\...,0
1,train_pair_000001,c,train,027.c,050.c,data\train\c\027.c,data\train\c\050.c,\n\n\n#include <stdio.h>\n\n#include <stdlib.h...,#include <stdio.h>\n#include <stdlib.h>\n#incl...,0
2,train_pair_000002,java,train,183.java,185.java,data\train\java\183.java,data\train\java\185.java,\n\nimport java.awt.*;\nimport java.awt.event....,\n\nimport java.awt.*;\nimport java.awt.event....,1
3,train_pair_000003,java,train,039.java,128.java,data\train\java\039.java,data\train\java\128.java,\npublic class CasePasswords\n{\n\n \n s...,\nimport java.io.*;\nimport java.util.Vector;\...,0
4,train_pair_000004,c,train,043.c,064.c,data\train\c\043.c,data\train\c\064.c,\n\n\n#include <stdio.h>\n#include <stdlib.h>\...,#include <stdio.h>\n#include <stdlib.h>\n#incl...,1


### Verificación del balance por etiqueta

El conteo de label confirma que el dataset combinado mantiene la proporción esperada entre positivos y negativos.


In [57]:
train_pairs_raw["label"].value_counts()

label
0    110
1    110
Name: count, dtype: int64

### Verificación por lenguaje y etiqueta

Agrupar por lenguaje y etiqueta permite revisar si el balance global también tiene sentido dentro de cada lenguaje. Esto ayuda a detectar que un lenguaje domine demasiado el entrenamiento.


In [58]:
train_pairs_raw.groupby(["language", "label"]).size()

language  label
c         0        59
          1        26
java      0        51
          1        84
dtype: int64

### Exportación del dataset completo de pares

Se guarda el dataset combinado antes de dividirlo en entrenamiento y validación. Esta salida conserva todos los pares generados en una sola tabla.


In [59]:
train_pairs_raw.to_csv(
    PROCESSED_PATH / "01_train_pairs_raw.csv",
    index=False
)

## 17. División en entrenamiento y validación

El dataset de pares se separa en dos subconjuntos: entrenamiento, usado para ajustar el modelo, y validación, usado para medir el desempeño durante el desarrollo sin entrenar directamente con esos ejemplos.

Se usa estratificación por label para mantener la misma proporción de positivos y negativos en ambos subconjuntos. Esto evita que la validación quede desbalanceada por azar.


In [60]:
train_model_raw, val_model_raw = train_test_split(
    train_pairs_raw,
    test_size=0.2,
    random_state=42,
    stratify=train_pairs_raw["label"]
)

train_model_raw = train_model_raw.reset_index(drop=True)
val_model_raw = val_model_raw.reset_index(drop=True)

print("Train model:", train_model_raw.shape)
print("Validation model:", val_model_raw.shape)

Train model: (176, 10)
Validation model: (44, 10)


### Revisión de distribución en train y validation

Se imprime la cantidad de positivos y negativos en cada partición para confirmar que la estratificación funcionó correctamente.


In [61]:
print("Train:")
print(train_model_raw["label"].value_counts())

print("\nValidation:")
print(val_model_raw["label"].value_counts())

Train:
label
0    88
1    88
Name: count, dtype: int64

Validation:
label
1    22
0    22
Name: count, dtype: int64


## 18. Exportación final para modelado

Se guardan los archivos finales que usará la siguiente etapa del proyecto: 01_train_model_raw.csv para entrenar y 01_val_model_raw.csv para validar.

Ambos conservan el código fuente raw para que el preprocesamiento pueda definirse después.


In [62]:
train_model_raw.to_csv(
    PROCESSED_PATH / "01_train_model_raw.csv",
    index=False
)

val_model_raw.to_csv(
    PROCESSED_PATH / "01_val_model_raw.csv",
    index=False
)

## 19. Validación final de integridad

La última celda revisa que las columnas indispensables existan y que no haya código faltante en los pares de entrenamiento o validación.

Estas aserciones funcionan como una prueba rápida de calidad: si alguna condición falla, el notebook se detiene y señala el problema antes de pasar a la etapa de preprocesamiento o modelado.


In [63]:
required_columns = [
    "pair_id",
    "language",
    "scenario",
    "file_1",
    "file_2",
    "code_1_raw",
    "code_2_raw",
    "label"
]

for col in required_columns:
    assert col in train_model_raw.columns, f"Falta columna en train: {col}"
    assert col in val_model_raw.columns, f"Falta columna en validation: {col}"

assert train_model_raw["code_1_raw"].notna().all()
assert train_model_raw["code_2_raw"].notna().all()
assert val_model_raw["code_1_raw"].notna().all()
assert val_model_raw["code_2_raw"].notna().all()

print("Dataset raw listo para pasar a preprocesado.")

Dataset raw listo para pasar a preprocesado.
